In [1]:
import pandas as pd
from datasets import load_dataset

# ── 1. Load the CNN/DailyMail train split ────────────────────────────────────
print("Loading CNN/DailyMail train split...")
ds = load_dataset("abisee/cnn_dailymail", "1.0.0")
train = ds["train"].to_pandas()
# 'id' column is the article hash, already present in the dataset
train = train[["id"]]

# ── 2. Load sentiment_checkpoint.csv ─────────────────────────────────────────
print("Loading sentiment_checkpoint.csv...")
sentiment = pd.read_csv("sentiment_checkpoint.csv")
# No id column — match to train split by index, then grab the hash id
sentiment["id"] = train["id"].values

# ── 3. Load crisis_scores_final.csv ──────────────────────────────────────────
print("Loading crisis_scores_final.csv...")
crisis = pd.read_csv("crisis_scores_final.csv")
# id column is the article hash, already present

# ── 4. Load ner_results_final.parquet ────────────────────────────────────────
print("Loading ner_results_final.parquet...")
ner = pd.read_parquet("ner_results_final.parquet")
# id column is the article hash, already present

# Normalise country name for the later join
ner["country_lower"] = ner["primary_country"].str.lower().str.strip()

# ── 5. Load StructuralFactorsOfCountries.csv ─────────────────────────────────
print("Loading StructuralFactorsOfCountries.csv...")
structural = pd.read_csv("StructuralFactorsOfCountries.csv")

# Identify the country column (adjust name if needed)
country_col = structural.columns[0]          # assume first column is country
structural["country_lower"] = structural[country_col].str.lower().str.strip()

# ── 6. Merge everything on id ────────────────────────────────────────────────
print("Merging tables...")

df = (
    train
    .merge(sentiment, on="id", how="inner", suffixes=("", "_sentiment"))
    .merge(crisis,    on="id", how="inner", suffixes=("", "_crisis"))
    .merge(ner,       on="id", how="inner", suffixes=("", "_ner"))
)

# ── 7. Join structural factors on country (lowercase) ────────────────────────
df = df.merge(structural, on="country_lower", how="left", suffixes=("", "_structural"))

# Drop helper columns
df = df.drop(columns=["country_lower"])

# ── 8. Save ──────────────────────────────────────────────────────────────────
output_path = "merged_dataset.parquet"
print(f"Saving to {output_path}...")
df.to_parquet(output_path, index=False)

print(f"Done! Final table shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

c:\Users\czajk\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading CNN/DailyMail train split...
Loading sentiment_checkpoint.csv...
Loading crisis_scores_final.csv...
Loading ner_results_final.parquet...
Loading StructuralFactorsOfCountries.csv...
Merging tables...
Saving to merged_dataset.parquet...
Done! Final table shape: (287113, 17)
Columns: ['id', 'sentiment_label', 'sentiment_score', 'crisis_density', 'crisis_conflict', 'crisis_disaster', 'crisis_poverty', 'crisis_health', 'crisis_displacement', 'crisis_instability', 'primary_country', 'country_freq', 'iso2', 'country', 'global_north_or_south', 'trade_value', 'HDI']
